In [ ]:
%pip install -e ../../../
%pip install scipy
%pip install plotly
%pip install nbformat

In [ ]:
import io

import numpy as np
from numpy.random import default_rng
RNG = default_rng(12806)
import pandas as pd
from scipy.optimize import curve_fit

from water_futures_battle.core import Settings
from water_futures_battle.jurisdictions import build_state

from utils import short_term_prediction, long_term_prediction

# Upload the necessary data
settings_desc = {
    "start_year": 2000,
    "end_year": 2024,
    "national_budget": 1000000000,
    "lifeline_volume": 80,
    "seed": 128
}
nl_desc = {
    'name': 'Netherlands',
    'id': 'NL0000',
    'jurisdictions-static_properties': 'jurisdictions-static_properties.xlsx',
    'municipalities-dynamic_properties': 'municipalities-dynamic_properties.xlsx'
}
netherlands = build_state(
    nl_desc,
    data_path='../../jurisdictions/',
    settings=Settings.from_config(settings_desc)
)
n_businesses = list(netherlands.municipalities)[0]._dynamic_properties['n_businesses'].sum(axis=1, skipna=True)
n_businesses.name = 'NL0000'

In [ ]:
# From the Vewin report we observed the national consumption and by dividing it
# by its number of units (population or businesses) we will get ther per unit demand
# for the years from 2000 to 2025.

# However, for the future, the participants will get only the min and max bound
# and the per unit demand will be sampled within those bounds.
# A deterministic value (observation) will have min and max bound equal

# Manually recorded data from the Vewin report on per business billed consumption (Millions of M^3/year)
data = """
year demand
2000 316,53
2001 322,73
2002 322,28
2003 317,56
2004 309,51
2005 300,04
2006 297,22
2007 297,72
2008 302,96
2009 305,36
2010 304,43
2011 300,19
2012 291,19
2013 289,79
2014 286,49
2015 286,52
2016 289,86
2017 292,26
2018 299,88
2019 307,97
2020 305,62
"""

df_dyn = pd.read_csv(
    io.StringIO(data), 
    sep='\s+', 
    decimal=',', 
    index_col='year'
)

df_dyn.index = pd.to_datetime(df_dyn.index, format='%Y')
df_dyn = df_dyn*1_000_000 # [m^3/year]

# Determinstic series of per capita demand between 2000 and 2020 [m^3/user/year]
per_user_demand = df_dyn['demand'] / n_businesses
per_user_demand = per_user_demand.dropna()
# Convert to daily values # [m^3/user/day]
days_in_year = per_user_demand.index.map(lambda d: 366 if d.is_leap_year else 365)
per_user_demand = per_user_demand / days_in_year # [m^3/user/day]
per_user_demand.name = 'NL0000'

historical_data = per_user_demand.to_frame()
historical_data['t'] = (historical_data.index.year - historical_data.index.year.min()).to_numpy()

# --- 2. Exponential Fitting Functions ---
def exponential_func(t, a, b):
    """Exponential curve function: y = a * exp(b * t)"""
    return a * np.exp(b * t)

def fit_exponential(df: pd.DataFrame) -> tuple[float, float]:
    """Fits an exponential curve to the sampled data."""
    # Fit the curve to the data
    params, _ = curve_fit(
        exponential_func,
        df['t'],
        df['NL0000'],
        p0=[300, 0.01] # Initial guess for a and b
    )
    return params


df_2021_2024 = short_term_prediction(
    historical_df=historical_data,
    n_future_years=4,
    n_samples=100,
    rng=RNG,
    fitting_function=fit_exponential,
    actual_function=exponential_func
)

df_post2024 = long_term_prediction(
    df=df_2021_2024,
    step_years=5,
    n_steps=3,
    uncertainty_bounds_variation=(0.95, 1.05) # 5% extra on both directions
)

# Merge the 3 dataframes (historical, short term uncertain prediction, long term uncertain prediction)
pbd_df = pd.concat([historical_data['NL0000'],historical_data['NL0000']], axis=1)
pbd_df.columns = df_2021_2024.columns
pbd_df = pd.concat([pbd_df, df_2021_2024, df_post2024], axis=0)
pbd_df.index.name = 'timestamp'
pbd_df = pbd_df/24

In [ ]:
# Manually calculated demands from the Vewin report to reproduce the same figure
# L/house/day
calculated_phdem = """
339,38
330,83
324,42
326,56
325,95
317,71
316,49
312,82
307,64
304,89
301,23
298,48
290,24
289,32
285,66
285,36
287,49
288,71
292,68
291,16
292,38
"""
# Create a dataframe with column NL000, index the years 
# also we convert the text above to float and from L/house/day to M^3/hour/house
historical_phdem_data = pd.DataFrame(
    data=[float(line.replace(',', '.'))*0.001/24 for line in calculated_phdem.strip().splitlines()],
    index=pd.date_range(start='2000-01-01', end='2020-01-01', freq='YS', name='timestamp'),
    columns=['NL0000']
)

historical_phdem_data['t'] = (historical_phdem_data.index.year - historical_phdem_data.index.year.min()).to_numpy()

def linear_func(t, m, c):
    return m * t + c

def fit_linear(df: pd.DataFrame) -> tuple[float, float]:
    params, _ = curve_fit(
        linear_func,
        df['t'],
        df['NL0000'],
        p0=[0.001, 0.01] # Initial guess for slope (m) and intercept (c)
    )
    return params

df_2021_2024 = short_term_prediction(
    historical_df=historical_phdem_data.iloc[-12:, :],
    n_future_years=4,
    n_samples=1,
    rng=RNG,
    fitting_function=fit_linear,
    actual_function=linear_func
)

df_post2024 = long_term_prediction(
    df=df_2021_2024,
    step_years=5,
    n_steps=3,
    uncertainty_bounds_variation=(0.95,1.03)
)

# Merge the 3 dataframes (historical, short term uncertain prediction, long term uncertain prediction)
phd_df = pd.concat([historical_phdem_data['NL0000'],historical_phdem_data['NL0000']], axis=1)
phd_df.columns = df_2021_2024.columns
phd_df = pd.concat([phd_df, df_2021_2024, df_post2024], axis=0)
phd_df.index.name = 'timestamp'

In [ ]:
from utils import plot_unit_demands
plot_unit_demands(phd_df, pbd_df).show()

In [ ]:
from water_futures_battle.core.base_model import DynamicProperties

wdm_dps = DynamicProperties('water_demand_model-dynamic_properties')

wdm_dps['per_business_demand'] = pbd_df.round(5) # so that it has two decimal digit in L
wdm_dps['per_house_demand'] = phd_df.round(5)

wdm_dps.dump('..')